# Phase 6: RAG & Vector Databases
## Day 29: LangChainRetrievalQA

Date: 2026-04-24

### Learning objectives
- Understand the LangChain RetrievalQA pattern.
- Learn document loaders, prompt templates, retrievers, and chains.
- Build a local RetrievalQA pipeline from scratch.
- Add source citations from chunk metadata.
- Understand simple chat memory.
- Compare LangChain-style components with plain Python.

In [ ]:
import json
import re
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    TfidfVectorizer = None
    SKLEARN_AVAILABLE = False

try:
    import langchain
    LANGCHAIN_AVAILABLE = True
except Exception:
    LANGCHAIN_AVAILABLE = False

def show(title, content):
    print("
" + "=" * 84)
    print(title)
    print("=" * 84)
    print(textwrap.dedent(str(content)).strip())

print("Setup complete.")
print("scikit-learn available:", SKLEARN_AVAILABLE)
print("LangChain available:", LANGCHAIN_AVAILABLE)

In [ ]:
raw_docs = [
    {
        "doc_id": "DOC-CAMPAIGN",
        "title": "Campaign Performance Notes",
        "text": '''
        Spring Coffee Push was an Instagram campaign for Berlin customers.
        The campaign spent 1200 EUR and produced 3420 clicks.
        It generated 184 conversions after a limited-time discount was added.
        Recommendation: scale the campaign carefully and monitor cost per conversion.

        Bank App Onboarding was an email campaign for existing banking users.
        It spent 800 EUR and produced 980 clicks.
        It generated only 42 conversions.
        The subject line was too generic and the call to action was not clear.
        Recommendation: rewrite the subject line and test a simpler onboarding message.

        Yoga Studio Trial was a TikTok campaign for beginners in Berlin.
        It spent 650 EUR and produced 2100 clicks.
        It generated 165 conversions.
        Short videos with beginner-friendly copy performed best.
        Recommendation: create more short videos and test new landing page copy.
        '''
    },
    {
        "doc_id": "DOC-OCR",
        "title": "OCR Pipeline Notes",
        "text": '''
        The OCR pipeline starts with a document image.
        OpenCV preprocessing improves OCR quality with grayscale conversion, denoising, thresholding, and deskewing.
        Tesseract works well on clean scans and simple layouts.
        EasyOCR can be useful for natural images and multilingual text.
        After OCR, raw text should be cleaned before structured extraction.

        The LLM extraction step receives OCR text and a JSON schema.
        The prompt should request valid JSON only.
        The extracted record should be validated with field types and business rules.
        If JSON parsing fails, the pipeline can repair small formatting issues.
        If validation fails, the pipeline can retry with the error message.
        '''
    },
    {
        "doc_id": "DOC-RAG",
        "title": "RAG Design Notes",
        "text": '''
        A RAG system retrieves relevant document chunks before generating an answer.
        Chunking strategy affects retrieval quality, answer accuracy, and context length.
        Very large chunks can contain too many ideas and reduce precision.
        Very small chunks can lose important context.
        Overlap helps preserve information near chunk boundaries.

        Embeddings turn text chunks into vectors.
        A vector database stores embeddings and supports similarity search.
        Retrieval quality should be evaluated with test queries and expected documents.
        Bad retrieval usually causes weak answers even if the language model is strong.
        Good RAG systems keep source metadata so answers can cite the right document.
        '''
    }
]

for doc in raw_docs:
    print(doc["doc_id"], "-", doc["title"], "-", len(doc["text"]), "characters")

## 1. RetrievalQA pattern

RetrievalQA means: retrieve useful context first, then answer with that context.

This is the core pattern behind many RAG apps.

In [ ]:
retrieval_qa_flow = pd.DataFrame([
    {"step": 1, "component": "Loader", "job": "Load raw documents"},
    {"step": 2, "component": "Splitter", "job": "Split documents into chunks"},
    {"step": 3, "component": "Embeddings", "job": "Convert chunks into vectors"},
    {"step": 4, "component": "Vector store", "job": "Store and search chunk vectors"},
    {"step": 5, "component": "Retriever", "job": "Return top matching chunks for a query"},
    {"step": 6, "component": "Prompt template", "job": "Combine question and context"},
    {"step": 7, "component": "LLM", "job": "Generate answer from retrieved context"},
])
retrieval_qa_flow

In [ ]:
langchain_shape = '''
LangChain-style shape:
loader = TextLoader("file.txt")
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(documents)
vectorstore = Chroma.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)
answer = qa_chain.invoke({"query": "What improved OCR quality?"})
'''
print(langchain_shape)

## 2. Document loader concept

A loader turns raw files into document objects.

Each document usually has page content and metadata.

In [ ]:
def simple_loader(raw_docs):
    loaded = []
    for doc in raw_docs:
        loaded.append({
            "page_content": textwrap.dedent(doc["text"]).strip(),
            "metadata": {
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "source": f'{doc["doc_id"]}.txt'
            }
        })
    return loaded

documents = simple_loader(raw_docs)
pprint(documents[0]["metadata"])
print("
Content preview:")
print(documents[0]["page_content"][:250])

In [ ]:
loader_types = pd.DataFrame([
    {"loader": "TextLoader", "input": ".txt files", "typical_use": "Simple notes and docs"},
    {"loader": "CSVLoader", "input": ".csv files", "typical_use": "Rows as documents"},
    {"loader": "PyPDFLoader", "input": ".pdf files", "typical_use": "PDF pages"},
    {"loader": "WebBaseLoader", "input": "Web pages", "typical_use": "Public page content"},
    {"loader": "DirectoryLoader", "input": "Folder", "typical_use": "Batch loading many files"},
])
loader_types

## 3. Text splitting

LangChain often uses `RecursiveCharacterTextSplitter`.

We will build a small local version that keeps metadata.

In [ ]:
def recursive_split_text(text, chunk_size=420, separators=None):
    separators = separators or ["

", "
", ". ", " ", ""]
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    separator = separators[0]
    remaining = separators[1:]
    if separator == "":
        return [text[i:i + chunk_size].strip() for i in range(0, len(text), chunk_size) if text[i:i + chunk_size].strip()]
    parts = text.split(separator)
    if len(parts) == 1:
        return recursive_split_text(text, chunk_size=chunk_size, separators=remaining)
    chunks, current = [], ""
    for part in parts:
        if not part.strip():
            continue
        candidate = (current + separator + part).strip() if current else part.strip()
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining))
            current = part.strip()
    if current:
        chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining))
    return chunks

def split_documents(documents, chunk_size=420):
    chunks = []
    for doc in documents:
        pieces = recursive_split_text(doc["page_content"], chunk_size=chunk_size)
        for i, piece in enumerate(pieces):
            chunks.append({
                "page_content": piece,
                "metadata": {
                    **doc["metadata"],
                    "chunk_index": i,
                    "chunk_id": f'{doc["metadata"]["doc_id"]}-{i:03d}'
                }
            })
    return chunks

chunks = split_documents(documents, chunk_size=420)
print("Number of chunks:", len(chunks))
pprint(chunks[0]["metadata"])
print(chunks[0]["page_content"])

In [ ]:
chunk_table = pd.DataFrame([
    {
        "chunk_id": chunk["metadata"]["chunk_id"],
        "doc_id": chunk["metadata"]["doc_id"],
        "title": chunk["metadata"]["title"],
        "chars": len(chunk["page_content"]),
        "preview": chunk["page_content"][:90] + "..."
    }
    for chunk in chunks
])
chunk_table

## 4. Embeddings and vector store

A vector store embeds chunks and supports similarity search.

We use TF-IDF or a tiny fallback so the notebook runs locally.

In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.documents = []
        self.matrix = None
        self.vectorizer = None
        self.vocabulary = None

    def _fallback_embedding(self, text):
        words = re.findall(r"[a-z]+", text.lower())
        return np.array([words.count(term) for term in self.vocabulary], dtype=float)

    def add_documents(self, documents):
        self.documents = documents
        texts = [doc["page_content"] for doc in documents]
        if SKLEARN_AVAILABLE:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.matrix = self.vectorizer.fit_transform(texts).toarray()
        else:
            self.vocabulary = sorted(set(re.findall(r"[a-z]+", " ".join(texts).lower())))
            self.matrix = np.vstack([self._fallback_embedding(text) for text in texts])

    def embed_query(self, query):
        if self.vectorizer is not None:
            return self.vectorizer.transform([query]).toarray()[0]
        return self._fallback_embedding(query)

    @staticmethod
    def cosine_similarity(a, b):
        denominator = np.linalg.norm(a) * np.linalg.norm(b)
        if denominator == 0:
            return 0.0
        return float(np.dot(a, b) / denominator)

    def similarity_search(self, query, k=3, metadata_filter=None):
        query_vector = self.embed_query(query)
        scores = [self.cosine_similarity(query_vector, row) for row in self.matrix]
        rows = []
        for doc, score in zip(self.documents, scores):
            if metadata_filter:
                keep = all(doc["metadata"].get(key) == value for key, value in metadata_filter.items())
                if not keep:
                    continue
            rows.append({"page_content": doc["page_content"], "metadata": doc["metadata"], "score": score})
        return sorted(rows, key=lambda row: row["score"], reverse=True)[:k]

vectorstore = SimpleVectorStore()
vectorstore.add_documents(chunks)
results = vectorstore.similarity_search("Which campaign had a generic subject line?", k=3)
pprint(results[0]["metadata"])
print("Score:", round(results[0]["score"], 4))
print(results[0]["page_content"])

## 5. Retriever

A retriever is a simple interface over search.

It hides vector store details and returns relevant documents.

In [ ]:
class SimpleRetriever:
    def __init__(self, vectorstore, k=3, metadata_filter=None):
        self.vectorstore = vectorstore
        self.k = k
        self.metadata_filter = metadata_filter

    def get_relevant_documents(self, query):
        return self.vectorstore.similarity_search(query, k=self.k, metadata_filter=self.metadata_filter)

retriever = SimpleRetriever(vectorstore, k=3)
retrieved_docs = retriever.get_relevant_documents("How can OCR quality be improved?")

for item in retrieved_docs:
    print("
Score:", round(item["score"], 4))
    print("Source:", item["metadata"]["chunk_id"], "|", item["metadata"]["title"])
    print(item["page_content"][:250])

In [ ]:
rag_only_retriever = SimpleRetriever(vectorstore, k=2, metadata_filter={"doc_id": "DOC-RAG"})
rag_results = rag_only_retriever.get_relevant_documents("Why does overlap matter?")
for item in rag_results:
    print(item["metadata"]["chunk_id"], item["score"])
    print(item["page_content"][:180])

## 6. Prompt templates

A prompt template combines instructions, retrieved context, and the user question.

Good templates tell the model to use only the provided context.

In [ ]:
class PromptTemplate:
    def __init__(self, template):
        self.template = template
    def format(self, **kwargs):
        return self.template.format(**kwargs)

qa_template_text = '''
You are a careful data science tutor.

Use only the context below to answer the question.
If the answer is not in the context, say: "I do not know from the provided context."

Context:
{context}

Question:
{question}

Answer:
- Give a short answer.
- Include source chunk IDs.
'''.strip()
qa_prompt_template = PromptTemplate(qa_template_text)

def format_context(retrieved_docs):
    blocks = []
    for item in retrieved_docs:
        metadata = item["metadata"]
        blocks.append(f'Source: {metadata["chunk_id"]} | {metadata["title"]}
{item["page_content"]}')
    return "

---

".join(blocks)

question = "Which campaign had a generic subject line?"
context = format_context(retriever.get_relevant_documents(question))
prompt = qa_prompt_template.format(context=context, question=question)
show("Formatted QA prompt", prompt)

## 7. Mock LLM answer

In production, this prompt would be sent to an LLM.

Here we use a simple rule-based mock answer so the notebook runs without API keys.

In [ ]:
def mock_llm_answer(question, retrieved_docs):
    question_lower = question.lower()
    context = "
".join(item["page_content"] for item in retrieved_docs)
    sources = [item["metadata"]["chunk_id"] for item in retrieved_docs]

    if "generic subject" in question_lower and "Bank App Onboarding" in context:
        return {"answer": "Bank App Onboarding had a generic subject line.", "sources": sources}
    if ("ocr quality" in question_lower or "improved" in question_lower) and "OpenCV preprocessing" in context:
        return {"answer": "OCR quality can be improved with grayscale conversion, denoising, thresholding, and deskewing.", "sources": sources}
    if "overlap" in question_lower and "Overlap helps" in context:
        return {"answer": "Overlap helps preserve information near chunk boundaries.", "sources": sources}
    if ("vector database" in question_lower or "rag" in question_lower) and ("vector database" in context or "RAG system" in context):
        return {"answer": "A RAG system retrieves relevant chunks from a vector database before generating an answer.", "sources": sources}
    if "clean scans" in question_lower and "Tesseract works well on clean scans" in context:
        return {"answer": "Tesseract works well on clean scans and simple layouts.", "sources": sources}
    return {"answer": "I do not know from the provided context.", "sources": sources}

answer = mock_llm_answer(question, retriever.get_relevant_documents(question))
pprint(answer)

## 8. RetrievalQA chain from scratch

A chain connects retriever, prompt template, and LLM.

This is the same idea as LangChain RetrievalQA, but written in plain Python.

In [ ]:
class SimpleRetrievalQA:
    def __init__(self, retriever, prompt_template, llm_function):
        self.retriever = retriever
        self.prompt_template = prompt_template
        self.llm_function = llm_function

    def invoke(self, query):
        retrieved_docs = self.retriever.get_relevant_documents(query)
        context = format_context(retrieved_docs)
        prompt = self.prompt_template.format(context=context, question=query)
        llm_result = self.llm_function(query, retrieved_docs)
        return {
            "query": query,
            "prompt": prompt,
            "answer": llm_result["answer"],
            "source_documents": retrieved_docs,
            "source_chunk_ids": llm_result["sources"]
        }

qa_chain = SimpleRetrievalQA(retriever=retriever, prompt_template=qa_prompt_template, llm_function=mock_llm_answer)
qa_result = qa_chain.invoke("How can OCR quality be improved?")
print("Answer:")
print(qa_result["answer"])
print("
Sources:")
print(qa_result["source_chunk_ids"])

In [ ]:
questions = [
    "Which campaign had a generic subject line?",
    "How can OCR quality be improved?",
    "Why does overlap matter?",
    "What does a vector database do in RAG?"
]
for q in questions:
    result = qa_chain.invoke(q)
    print("
Question:", q)
    print("Answer:", result["answer"])
    print("Sources:", result["source_chunk_ids"])

## 9. Source citations

RAG answers should show where the answer came from.

Source metadata makes the answer easier to verify.

In [ ]:
def format_answer_with_sources(result):
    source_lines = []
    for doc in result["source_documents"]:
        metadata = doc["metadata"]
        source_lines.append(f'- {metadata["chunk_id"]}: {metadata["title"]}')
    return result["answer"] + "

Sources:
" + "
".join(source_lines)

formatted = format_answer_with_sources(qa_chain.invoke("Which campaign had a generic subject line?"))
print(formatted)

In [ ]:
def get_unique_sources(result):
    seen = set()
    sources = []
    for doc in result["source_documents"]:
        metadata = doc["metadata"]
        key = (metadata["doc_id"], metadata["title"])
        if key not in seen:
            seen.add(key)
            sources.append({"doc_id": metadata["doc_id"], "title": metadata["title"]})
    return sources

pprint(get_unique_sources(qa_chain.invoke("How can OCR quality be improved?")))

## 10. Chat memory concept

Memory keeps previous conversation turns.

For RAG, memory can help interpret follow-up questions, but it should not replace retrieval.

In [ ]:
class SimpleConversationMemory:
    def __init__(self, max_turns=4):
        self.max_turns = max_turns
        self.turns = []

    def add_turn(self, question, answer):
        self.turns.append({"question": question, "answer": answer})
        self.turns = self.turns[-self.max_turns:]

    def get_history_text(self):
        if not self.turns:
            return ""
        lines = []
        for turn in self.turns:
            lines.append(f'User: {turn["question"]}')
            lines.append(f'Assistant: {turn["answer"]}')
        return "
".join(lines)

memory = SimpleConversationMemory(max_turns=3)
first = qa_chain.invoke("How can OCR quality be improved?")
memory.add_turn(first["query"], first["answer"])
second = qa_chain.invoke("Which tool works well on clean scans?")
memory.add_turn(second["query"], second["answer"])
print(memory.get_history_text())

In [ ]:
memory_template_text = '''
You are a careful data science tutor.

Conversation history:
{history}

Use only the context below to answer the latest question.
If the answer is not in the context, say: "I do not know from the provided context."

Context:
{context}

Latest question:
{question}

Answer:
'''.strip()
memory_prompt_template = PromptTemplate(memory_template_text)

def build_memory_prompt(question, retrieved_docs, memory):
    return memory_prompt_template.format(
        history=memory.get_history_text(),
        context=format_context(retrieved_docs),
        question=question
    )

follow_up_question = "What about natural images?"
follow_up_docs = retriever.get_relevant_documents(follow_up_question)
follow_up_prompt = build_memory_prompt(follow_up_question, follow_up_docs, memory)
show("Prompt with memory", follow_up_prompt)

## 11. Retrieval evaluation

Before building a chatbot UI, evaluate retrieval.

If retrieval is bad, the answer will probably be bad too.

In [ ]:
test_cases = [
    {"query": "generic subject line email campaign", "expected_doc_ids": {"DOC-CAMPAIGN"}},
    {"query": "OpenCV grayscale thresholding deskewing", "expected_doc_ids": {"DOC-OCR"}},
    {"query": "overlap near chunk boundaries", "expected_doc_ids": {"DOC-RAG"}},
    {"query": "vector database stores embeddings", "expected_doc_ids": {"DOC-RAG"}},
]

def evaluate_retriever(retriever, test_cases):
    rows = []
    for case in test_cases:
        docs = retriever.get_relevant_documents(case["query"])
        retrieved_doc_ids = {doc["metadata"]["doc_id"] for doc in docs}
        hit = len(retrieved_doc_ids & case["expected_doc_ids"]) > 0
        rows.append({
            "query": case["query"],
            "expected": sorted(case["expected_doc_ids"]),
            "retrieved": sorted(retrieved_doc_ids),
            "hit": hit
        })
    return pd.DataFrame(rows)

retrieval_eval = evaluate_retriever(retriever, test_cases)
retrieval_eval

In [ ]:
hit_rate = retrieval_eval["hit"].mean()
print("Retriever hit rate:", round(hit_rate, 3))

## 12. LangChain mapping

Even if you use plain Python first, it helps to know how the pieces map to LangChain.

The concept is more important than the library.

In [ ]:
mapping = pd.DataFrame([
    {"plain_python": "simple_loader", "langchain_style": "TextLoader, CSVLoader, PyPDFLoader"},
    {"plain_python": "split_documents", "langchain_style": "RecursiveCharacterTextSplitter"},
    {"plain_python": "SimpleVectorStore", "langchain_style": "Chroma, FAISS, Pinecone, Weaviate"},
    {"plain_python": "SimpleRetriever", "langchain_style": "vectorstore.as_retriever()"},
    {"plain_python": "PromptTemplate", "langchain_style": "ChatPromptTemplate or PromptTemplate"},
    {"plain_python": "SimpleRetrievalQA", "langchain_style": "RetrievalQA or LCEL chain"},
    {"plain_python": "SimpleConversationMemory", "langchain_style": "ConversationBufferMemory or message history"},
])
mapping

In [ ]:
langchain_install_notes = '''
Modern LangChain installs can be split into packages.

Common installs:
pip install langchain
pip install langchain-community
pip install langchain-openai
pip install chromadb
pip install faiss-cpu

Because LangChain changes often, always check the current docs for imports.
The concept stays stable:
load -> split -> embed -> store -> retrieve -> prompt -> answer
'''
print(langchain_install_notes)

## Tricky bits

RetrievalQA failures are usually retrieval failures.

Always inspect retrieved chunks before blaming the LLM.

In [ ]:
tricky_bits = pd.DataFrame([
    {"problem": "The answer is wrong", "likely_cause": "Retriever returned weak context", "fix": "Inspect top chunks and improve chunking or embeddings"},
    {"problem": "The answer has no sources", "likely_cause": "Metadata was not stored or returned", "fix": "Store chunk_id, doc_id, title, and source"},
    {"problem": "Follow-up questions are confusing", "likely_cause": "No memory or query rewriting", "fix": "Add memory or rewrite follow-up questions into standalone queries"},
    {"problem": "Model invents details", "likely_cause": "Prompt does not restrict answer to context", "fix": "Tell the model to use only provided context"},
    {"problem": "Good docs are not retrieved", "likely_cause": "Bad chunks, wrong embedding model, or top_k too low", "fix": "Evaluate retrieval and tune settings"},
])
tricky_bits

In [ ]:
def diagnose_retrievalqa_issue(symptom):
    symptom = symptom.lower()
    if "wrong" in symptom or "invent" in symptom:
        return "Inspect retrieved chunks and strengthen the prompt to use only context."
    if "no source" in symptom or "citation" in symptom:
        return "Check that metadata is stored and returned."
    if "follow" in symptom:
        return "Add memory or rewrite the follow-up as a standalone question."
    if "not retrieved" in symptom or "missing" in symptom:
        return "Tune chunking, embeddings, top_k, and retrieval filters."
    return "Inspect retrieval results, prompt, and answer formatting."

for symptom in ["The answer is wrong", "There is no source citation", "Follow-up questions fail", "Relevant chunk is missing"]:
    print(symptom, "=>", diagnose_retrievalqa_issue(symptom))

## Trick questions

1. Is RetrievalQA only about the LLM?

<details><summary>Answer</summary>

No. Retrieval quality, chunking, metadata, and prompt design are just as important.

</details>

2. Why return source documents?

<details><summary>Answer</summary>

They make answers verifiable and easier to debug.

</details>

3. Should chat memory replace retrieval?

<details><summary>Answer</summary>

No. Memory helps with conversation context, but retrieval should still provide factual grounding.

</details>

4. What should the prompt say when context is missing?

<details><summary>Answer</summary>

It should tell the model to say that it does not know from the provided context.

</details>

5. What is the first thing to inspect when a RAG answer is bad?

<details><summary>Answer</summary>

Inspect the retrieved chunks.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Load raw documents with the simple loader.

loaded_docs = ___

assert isinstance(loaded_docs, list)
assert "page_content" in loaded_docs[0]
assert "metadata" in loaded_docs[0]
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Split loaded documents into chunks.

exercise_chunks = ___

assert isinstance(exercise_chunks, list)
assert len(exercise_chunks) > 0
assert "chunk_id" in exercise_chunks[0]["metadata"]
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Create a vector store and add chunks.

exercise_store = SimpleVectorStore()
___

assert exercise_store.matrix is not None
assert len(exercise_store.documents) == len(exercise_chunks)
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Create a retriever with k=2.

exercise_retriever = ___

assert exercise_retriever.k == 2
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Retrieve documents for an OCR question.

retrieved = ___

assert isinstance(retrieved, list)
assert len(retrieved) == 2
assert "score" in retrieved[0]
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Format context from retrieved documents.

context_text = ___

assert isinstance(context_text, str)
assert "Source:" in context_text
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Build a RetrievalQA chain.

exercise_chain = ___

assert hasattr(exercise_chain, "invoke")
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Ask the chain a question.

chain_result = ___

assert "answer" in chain_result
assert "source_documents" in chain_result
print("Exercise 8 passed.")

## Solutions

<details><summary>Exercise 1 solution</summary>

```python
loaded_docs = simple_loader(raw_docs)
```
</details>

<details><summary>Exercise 2 solution</summary>

```python
exercise_chunks = split_documents(loaded_docs, chunk_size=420)
```
</details>

<details><summary>Exercise 3 solution</summary>

```python
exercise_store.add_documents(exercise_chunks)
```
</details>

<details><summary>Exercise 4 solution</summary>

```python
exercise_retriever = SimpleRetriever(exercise_store, k=2)
```
</details>

<details><summary>Exercise 5 solution</summary>

```python
retrieved = exercise_retriever.get_relevant_documents("How can OCR quality be improved?")
```
</details>

<details><summary>Exercise 6 solution</summary>

```python
context_text = format_context(retrieved)
```
</details>

<details><summary>Exercise 7 solution</summary>

```python
exercise_chain = SimpleRetrievalQA(
    retriever=exercise_retriever,
    prompt_template=qa_prompt_template,
    llm_function=mock_llm_answer
)
```
</details>

<details><summary>Exercise 8 solution</summary>

```python
chain_result = exercise_chain.invoke("How can OCR quality be improved?")
```
</details>

## Cumulative review exercises

These mix topics from Days 19 to 28. Fill in `___` and run each cell.

In [ ]:
# Review 1: Structured output
json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___
assert parsed["clicks"] == 100
print("Review 1 passed.")

In [ ]:
# Review 2: Information extraction
record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___
assert abs(conversion_rate - 0.08) < 1e-9
print("Review 2 passed.")

In [ ]:
# Review 3: Tesseract basics
english_lang_code = ___
assert english_lang_code == "eng"
print("Review 3 passed.")

In [ ]:
# Review 4: EasyOCR
easyocr_languages = ___
assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 4 passed.")

In [ ]:
# Review 5: OpenCV preprocessing
threshold_method = ___
assert threshold_method.lower() == "adaptive"
print("Review 5 passed.")

In [ ]:
# Review 6: OCR plus LLM pipeline
structured_format = ___
assert structured_format.upper() == "JSON"
print("Review 6 passed.")

In [ ]:
# Review 7: Document intelligence
quality_rule = ___
assert "total" in quality_rule.lower() or "date" in quality_rule.lower() or "id" in quality_rule.lower()
print("Review 7 passed.")

In [ ]:
# Review 8: Embeddings
a = np.array([1, 0, 0])
b = np.array([1, 1, 0])
score = ___
assert 0.70 < score < 0.72
print("Review 8 passed.")

In [ ]:
# Review 9: Chunking
sample_text = "A RAG system retrieves relevant chunks before generating an answer."
chunked = ___
assert isinstance(chunked, list)
assert len(chunked) > 0
print("Review 9 passed.")

In [ ]:
# Review 10: Chroma and FAISS
vector_db_stores = ___
assert "embedding" in vector_db_stores.lower() or "vector" in vector_db_stores.lower()
print("Review 10 passed.")

## Cumulative review solutions

<details><summary>Show solutions</summary>

```python
# Review 1
parsed = json.loads(json_text)

# Review 2
conversion_rate = record["conversions"] / record["clicks"]

# Review 3
english_lang_code = "eng"

# Review 4
easyocr_languages = ["en", "de"]

# Review 5
threshold_method = "adaptive"

# Review 6
structured_format = "JSON"

# Review 7
quality_rule = "Invoice total must be present and non-negative."

# Review 8
score = SimpleVectorStore.cosine_similarity(a, b)

# Review 9
chunked = recursive_split_text(sample_text, chunk_size=40)

# Review 10
vector_db_stores = "embeddings and metadata"
```

</details>

In [ ]:
cheat_sheet = '''
DAY 29 CHEAT SHEET: LANGCHAIN RETRIEVALQA

RetrievalQA pattern:
1. Load documents.
2. Split documents into chunks.
3. Embed chunks.
4. Store vectors.
5. Retrieve relevant chunks.
6. Format prompt with context and question.
7. Generate answer with LLM.
8. Return answer and sources.

LangChain concepts:
- Loader: reads files or data sources.
- Text splitter: creates chunks.
- Vector store: stores embeddings and metadata.
- Retriever: returns relevant chunks.
- Prompt template: formats context and question.
- Chain: connects retriever, prompt, and LLM.
- Memory: stores conversation history.

Good RAG prompt rules:
- Use only the provided context.
- Say when the answer is not in context.
- Keep answers short and clear.
- Return source chunk IDs.

Debug checklist:
- Inspect retrieved chunks.
- Check chunk metadata.
- Check prompt formatting.
- Check top_k.
- Evaluate retrieval with test queries.
'''
print(cheat_sheet)

## Next up: Day 30 — RagChatbotProject

You will build the final full RAG chatbot over synthetic campaign CSVs.